# Commodity Derivatives Intelligence: Step 1 Preprocessing

This notebook explains and executes **Step 1: Clean the Data** in a simple, step-by-step manner.

### The 4 MCX Gold Contracts:
- **GOLDM**: 100 g lot size, quoted per 10 g (995 purity)
- **GOLDTEN**: 10 g lot size, quoted per 10 g (999 purity)
- **GOLDGUINEA**: 8 g lot size, quoted per 8 g (999 purity)
- **GOLDPETAL**: 1 g lot size, quoted per 1 g (999 purity)

Our goal here is simple:
1. Load the 5 Bhavcopy CSV files
2. Standardize column names and parse dates
3. Convert every contract's close price to **`close_per_gram`**
4. Run 5 data quality checks and log any issues
5. Save the clean data to `out/gold_futures_clean.csv`

### Step 1.1: Setup Folders and Check Available Files

In [ ]:
from pathlib import Path
import pandas as pd

# Set input and output folder paths
data_folder = Path('data')
out_folder = Path('out')
out_folder.mkdir(parents=True, exist_ok=True)

# Find all CSV files in the data directory
csv_files = sorted(list(data_folder.glob('*.csv')))
print(f'Found {len(csv_files)} CSV files in {data_folder}:')
for f in csv_files:
    print(' -', f.name)

### Step 1.2: Preview a Raw CSV File
Let's inspect how the exchange provides the data in its raw format before cleaning.

In [ ]:
# Read and view the first 3 rows of GOLDTEN
sample_raw = pd.read_csv(data_folder / 'GOLDTEN_30OCT2026.csv')
sample_raw.head(3)

### Step 1.3: Load Each File, Strip Extra Spaces, and Filter
In this step, we loop over each file and:
1. Strip trailing/leading spaces from column headers.
2. Strip trailing spaces from values like `'GOLDTEN   '` -> `'GOLDTEN'`.
3. Keep only rows where `Instrument Name == 'FUTCOM'` (futures) and symbol is one of our 4 contracts.
4. Drop non-essential columns: `Instrument Name`, `Option Type`, `Strike Price`, and `Volume(In 000's)`.

In [ ]:
allowed_symbols = ['GOLDM', 'GOLDTEN', 'GOLDGUINEA', 'GOLDPETAL']
cols_to_drop = ['Instrument Name', 'Option Type', 'Strike Price', "Volume(In 000's)"]

loaded_tables = []

for file_path in csv_files:
    # Read raw CSV
    df = pd.read_csv(file_path)
    
    # 1. Strip spaces from column headers
    df.columns = df.columns.str.strip()
    
    # 2. Strip spaces from text columns
    df['Symbol'] = df['Symbol'].astype(str).str.strip()
    df['Instrument Name'] = df['Instrument Name'].astype(str).str.strip()
    df['Expiry Date'] = df['Expiry Date'].astype(str).str.strip()
    
    # 3. Filter for futures of our 4 gold symbols
    mask = (df['Instrument Name'] == 'FUTCOM') & (df['Symbol'].isin(allowed_symbols))
    df = df[mask].copy()
    
    # Record filename to trace back any issues during validation checks
    df['_source_file'] = file_path.name
    
    # 4. Drop columns we do not need
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])
    
    loaded_tables.append(df)

# Combine all loaded tables into one single DataFrame
df_all = pd.concat(loaded_tables, ignore_index=True)
print(f'Total combined rows: {len(df_all)}')
df_all.head(3)

### Step 1.4: Parse Dates and Rename Columns
We convert `Date` (e.g. `'01 Oct 2026'`) and `Expiry Date` (e.g. `'30OCT2026'`) into real datetime objects.
Then we rename all columns to simple, lowercase names.

In [ ]:
# Parse dates into datetime
df_all['Date'] = pd.to_datetime(df_all['Date'], format='%d %b %Y')
df_all['Expiry Date'] = pd.to_datetime(df_all['Expiry Date'], format='%d%b%Y')

# Rename to lowercase names
rename_map = {
    'Date': 'date',
    'Symbol': 'symbol',
    'Expiry Date': 'expiry_date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Previous Close': 'prev_close',
    'Volume(Lots)': 'volume_lots',
    'Value(Lacs)': 'value_lacs',
    'Open Interest(Lots)': 'open_interest_lots'
}
df_all = df_all.rename(columns=rename_map)

df_all.head(3)

### Step 1.5: Add 3 Essential Columns
1. **`close_per_gram`**:
   - **GOLDM**: quoted per 10 g -> `close / 10`
   - **GOLDTEN**: quoted per 10 g -> `close / 10`
   - **GOLDGUINEA**: quoted per 8 g -> `close / 8`
   - **GOLDPETAL**: quoted per 1 g -> `close / 1`
2. **`traded`**: `True` if `volume_lots > 0`, else `False` (zero-volume days mean no trade occurred).
3. **`days_to_expiry`**: Calendar days remaining (`expiry_date - date`).

In [ ]:
# Quote unit in grams for each contract
quote_units = {
    'GOLDM': 10,
    'GOLDTEN': 10,
    'GOLDGUINEA': 8,
    'GOLDPETAL': 1
}

# 1. Calculate price per gram
df_all['close_per_gram'] = df_all['close'] / df_all['symbol'].map(quote_units)

# 2. Mark whether the contract was traded today
df_all['traded'] = df_all['volume_lots'] > 0

# 3. Compute calendar days to expiry
df_all['days_to_expiry'] = (df_all['expiry_date'] - df_all['date']).dt.days

# Preview the added columns
df_all[['date', 'symbol', 'close', 'close_per_gram', 'volume_lots', 'traded', 'days_to_expiry']].head(6)

### Step 1.6: Run Data Quality Checks
We check for 5 types of potential anomalies and save any issues to `out/issues_log.csv`:
1. **Filename check**: Does filename match the symbol and expiry date in the file?
2. **Missing weekdays**: Are any Monday-to-Friday trading days missing?
3. **Duplicate rows**: Any duplicate `(date, symbol, expiry_date)` records?
4. **Invalid close**: Missing or <= 0 close price on a day that traded?
5. **High < Low**: Any inverted price ranges?

In [ ]:
issues = []

# Check 1: Filename matches symbol and expiry inside
for file_path in csv_files:
    stem = file_path.stem.upper()
    file_rows = df_all[df_all['_source_file'] == file_path.name]
    for _, row in file_rows[['symbol', 'expiry_date']].drop_duplicates().iterrows():
        exp_str = row['expiry_date'].strftime('%d%b%Y').upper()
        expected_stem = f"{row['symbol']}_{exp_str}"
        if stem != expected_stem:
            issues.append({
                'file': file_path.name,
                'problem': 'File name mismatch',
                'detail': f"File name '{stem}' does not match content '{expected_stem}'"
            })

# Check 2: Missing weekdays
for (sym, exp), group in df_all.groupby(['symbol', 'expiry_date']):
    all_weekdays = pd.bdate_range(group['date'].min(), group['date'].max())
    present_dates = set(group['date'])
    missing_days = sorted(set(all_weekdays) - present_dates)
    for m in missing_days:
        issues.append({
            'file': group['_source_file'].iloc[0],
            'problem': 'Missing weekday',
            'detail': f"{sym} ({exp.strftime('%Y-%m-%d')}): {m.strftime('%Y-%m-%d')} ({m.strftime('%A')}) missing"
        })

# Check 3: Duplicate rows
duplicates = df_all[df_all.duplicated(subset=['date', 'symbol', 'expiry_date'], keep=False)]
for _, row in duplicates.iterrows():
    issues.append({
        'file': row['_source_file'],
        'problem': 'Duplicate (date, symbol, expiry_date)',
        'detail': f"{row['symbol']} on {row['date'].strftime('%Y-%m-%d')}"
    })

# Check 4: Close missing or <= 0 on traded days
bad_close = df_all[df_all['traded'] & (df_all['close'].isna() | (df_all['close'] <= 0))]
for _, row in bad_close.iterrows():
    issues.append({
        'file': row['_source_file'],
        'problem': 'Invalid close on traded day',
        'detail': f"{row['symbol']} on {row['date'].strftime('%Y-%m-%d')}: close={row['close']}"
    })

# Check 5: High < Low
bad_high_low = df_all[df_all['high'] < df_all['low']]
for _, row in bad_high_low.iterrows():
    issues.append({
        'file': row['_source_file'],
        'problem': 'High < Low',
        'detail': f"{row['symbol']} on {row['date'].strftime('%Y-%m-%d')}: high={row['high']} < low={row['low']}"
    })

# Save issues log to CSV
issues_df = pd.DataFrame(issues, columns=['file', 'problem', 'detail'])
issues_df.to_csv(out_folder / 'issues_log.csv', index=False)

print(f'Validation complete! Total issues detected: {len(issues)}')
if len(issues) == 0:
    print('All checks passed with 0 issues!')
else:
    print(issues_df)

### Step 1.7: Sort, Deduplicate, and Save Clean Data
We drop the internal tracking column `_source_file`, sort by `symbol`, `expiry_date`, `date`, remove exact duplicates, and save `gold_futures_clean.csv`.

In [ ]:
# Drop tracking column
df_clean = df_all.drop(columns=['_source_file'])

# Sort and drop any exact duplicate rows
df_clean = df_clean.sort_values(by=['symbol', 'expiry_date', 'date']).drop_duplicates().reset_index(drop=True)

# Format date columns as clean YYYY-MM-DD strings for CSV export
df_clean_export = df_clean.copy()
df_clean_export['date'] = df_clean_export['date'].dt.strftime('%Y-%m-%d')
df_clean_export['expiry_date'] = df_clean_export['expiry_date'].dt.strftime('%Y-%m-%d')

# Save to CSV
output_file = out_folder / 'gold_futures_clean.csv'
df_clean_export.to_csv(output_file, index=False)
print(f'Successfully saved {len(df_clean_export)} clean rows to {output_file}')

### Step 1.8: Summary Table of All Contracts
Let's print the overview of dates, trading days, and average volume for each contract.

In [ ]:
summary_list = []
for (sym, exp), grp in df_clean.groupby(['symbol', 'expiry_date']):
    summary_list.append({
        'symbol': sym,
        'expiry_date': exp.strftime('%Y-%m-%d'),
        'first_date': grp['date'].min().strftime('%Y-%m-%d'),
        'last_date': grp['date'].max().strftime('%Y-%m-%d'),
        'number_of_days': len(grp),
        'traded_days': int(grp['traded'].sum()),
        'avg_volume_lots': round(grp['volume_lots'].mean(), 2)
    })

summary_df = pd.DataFrame(summary_list)
summary_df